# Apache Polaris — Metastore Schema Verification

**Date:** 2026-05-15  
**Reference:** https://polaris.apache.org/releases/1.3.0/metastores/

## 학습 목표 (Learning Goals)
- Polaris 메타스토어의 PostgreSQL 스키마 구조 직접 확인
- entities, grant_records, principal_secrets 테이블 분석
- type_code 의미 파악
- entity_version 실제 데이터로 검증

## 메타스토어 타입
| 타입 | 용도 | 특징 |
|------|------|------|
| in-memory | 개발/테스트 | 재시작 시 데이터 소실 |
| relational-jdbc | 운영 | PostgreSQL/H2, 영구 저장 |

In [5]:
# ============================================================
# Prerequisites
# ============================================================
# uv add psycopg2-binary pandas requests
# pip install psycopg2-binary pandas requests
# ============================================================

import subprocess
import sys
import os

REQUIRED = ['psycopg2-binary', 'pandas', 'requests', 'sqlalchemy']

def install(pkg):
    result = subprocess.run(
        ['uv', 'add', pkg],
        capture_output=True, text=True, cwd=os.getcwd()
    )
    if result.returncode == 0:
        return '✅', 'uv'
    result = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', pkg],
        capture_output=True, text=True
    )
    return ('✅', 'pip') if result.returncode == 0 else ('❌', 'failed')

print('Installing dependencies...')
for pkg in REQUIRED:
    status, method = install(pkg)
    print(f'  {status} {pkg} ({method})')

print('\n⚠️  Restart kernel after first install.')

Installing dependencies...
  ✅ psycopg2-binary (uv)
  ✅ pandas (uv)
  ✅ requests (uv)
  ✅ sqlalchemy (uv)

⚠️  Restart kernel after first install.


In [1]:
# ── src path bootstrap + shared config (modules live in /src) ───────────────────
import sys, pathlib
_ROOT = pathlib.Path.cwd()
while not (_ROOT / "src").is_dir() and _ROOT != _ROOT.parent:
    _ROOT = _ROOT.parent
_SRC = _ROOT / "src"
if str(_SRC) not in sys.path:
    sys.path.insert(0, str(_SRC))
from nb_support import *
init_env("local")   # default safe env; POLARIS_URL/REALM/ROOT_SECRET/MINIO_*/PG_* now set
# Cell 1 — Setup: PostgreSQL + Polaris connections
import psycopg2
import psycopg2.extras
import pandas as pd
import requests
import json
from IPython.display import display

# PostgreSQL connection (via port-forward or LoadBalancer)
PG_CONFIG = {
    'host':     PG_HOST,
    'port':     PG_PORT,
    'database': PG_DB,
    'user':     PG_USER,
    'password': PG_PASSWORD
}

# Polaris API
POLARIS_URL = POLARIS_URL  # from init_env (src/config)
REALM       = REALM  # from init_env

def pg_query(sql, params=None):
    conn = psycopg2.connect(**PG_CONFIG)
    df = pd.read_sql_query(sql, conn, params=params)
    conn.close()
    return df

def get_token():
    r = requests.post(
        f'{POLARIS_URL}/api/catalog/v1/oauth/tokens',
        headers={'Polaris-Realm': REALM},
        data={
            'grant_type': 'client_credentials',
            'client_id': 'root',
            'client_secret': ROOT_SECRET,
            'scope': 'PRINCIPAL_ROLE:ALL'
        }
    )
    return r.json()['access_token']

TOKEN = get_token()

from sqlalchemy import create_engine, text

PG_URL = f"postgresql://{PG_USER}:{PG_PASSWORD}@{PG_HOST}:{PG_PORT}/{PG_DB}"  # from init_env
engine = create_engine(PG_URL)

def pg_query(sql, params=None):
    with engine.connect() as conn:
        return pd.read_sql_query(text(sql), conn, params=params)

# Test
try:
    df = pg_query('SELECT version()')
    print(f'✅ PostgreSQL connected: {df.iloc[0,0][:50]}...')
except Exception as e:
    print(f'❌ Failed: {e}')

print(f'✅ Polaris connected')

✅ PostgreSQL connected: PostgreSQL 17.6 on aarch64-unknown-linux-gnu, comp...
✅ Polaris connected


## Schema Overview
Polaris가 bootstrap 시 자동으로 생성하는 `polaris_schema` 구조를 확인합니다.

In [2]:
# Cell 2 — List all tables in polaris_schema
print('=' * 60)
print('POLARIS SCHEMA TABLES')
print('=' * 60)

df = pg_query("""
    SELECT 
        table_name,
        pg_size_pretty(pg_total_relation_size(
            quote_ident(table_schema) || '.' || quote_ident(table_name)
        )) AS size
    FROM information_schema.tables
    WHERE table_schema = 'polaris_schema'
    ORDER BY table_name
""")
display(df)

print('\n📋 Column details per table:')
for table in df['table_name'].tolist():
    cols = pg_query(f"""
        SELECT column_name, data_type, is_nullable
        FROM information_schema.columns
        WHERE table_schema = 'polaris_schema'
        AND table_name = '{table}'
        ORDER BY ordinal_position
    """)
    print(f'\n  📁 {table}:')
    for _, row in cols.iterrows():
        nullable = '?' if row['is_nullable'] == 'YES' else '!'
        print(f'    {nullable} {row["column_name"]} ({row["data_type"]})')

POLARIS SCHEMA TABLES


,table_name,size
0,entities,80 kB
1,events,32 kB
2,grant_records,32 kB
3,policy_mapping_record,24 kB
4,principal_authentication_data,32 kB
5,version,32 kB



📋 Column details per table:

  📁 entities:
    ! realm_id (text)
    ! catalog_id (bigint)
    ! id (bigint)
    ! parent_id (bigint)
    ! name (text)
    ! entity_version (integer)
    ! type_code (integer)
    ! sub_type_code (integer)
    ! create_timestamp (bigint)
    ! drop_timestamp (bigint)
    ! purge_timestamp (bigint)
    ! to_purge_timestamp (bigint)
    ! last_update_timestamp (bigint)
    ! properties (jsonb)
    ! internal_properties (jsonb)
    ! grant_records_version (integer)
    ? location_without_scheme (text)

  📁 events:
    ! realm_id (text)
    ! catalog_id (text)
    ! event_id (text)
    ? request_id (text)
    ! event_type (text)
    ! timestamp_ms (bigint)
    ? principal_name (text)
    ! resource_type (text)
    ! resource_identifier (text)
    ! additional_properties (jsonb)

  📁 grant_records:
    ! realm_id (text)
    ! securable_catalog_id (bigint)
    ! securable_id (bigint)
    ! grantee_catalog_id (bigint)
    ! grantee_id (bigint)
    ! privilege

## entities table
모든 Polaris 엔티티(Realm, Principal, Catalog, Namespace, Table 등)가 저장되는 핵심 테이블

In [3]:
# Cell 3 — entities table deep dive
print('=' * 60)
print('ENTITIES TABLE')
print('=' * 60)

# type_code mapping
TYPE_CODES = {
    1: 'ROOT_CONTAINER',
    2: 'PRINCIPAL',
    3: 'PRINCIPAL_ROLE',
    4: 'CATALOG',
    5: 'NAMESPACE',
    6: 'TABLE_LIKE',
    7: 'CATALOG_ROLE',
    8: 'POLICY',
}

df = pg_query("""
    SELECT 
        realm_id,
        id,
        parent_id,
        name,
        type_code,
        sub_type_code,
        entity_version,
        create_timestamp,
        last_update_timestamp,
        properties,
        internal_properties
    FROM polaris_schema.entities
    ORDER BY type_code, create_timestamp
""")

# Add human readable type
df['entity_type'] = df['type_code'].map(TYPE_CODES)

print(f'\n📊 Total entities: {len(df)}')
print('\n📋 Entity summary by type:')
summary = df.groupby(['type_code', 'entity_type']).size().reset_index(name='count')
display(summary)

print('\n📋 All entities:')
display(df[['realm_id', 'name', 'entity_type', 'entity_version', 
            'parent_id', 'id', 'create_timestamp']])

ENTITIES TABLE

📊 Total entities: 11

📋 Entity summary by type:


,type_code,entity_type,count
0,1,ROOT_CONTAINER,1
1,2,PRINCIPAL,2
2,3,PRINCIPAL_ROLE,2
3,4,CATALOG,1
4,5,NAMESPACE,2
5,6,TABLE_LIKE,1
6,7,CATALOG_ROLE,1
7,8,POLICY,1



📋 All entities:


,realm_id,name,entity_type,entity_version,parent_id,id,create_timestamp
0,POLARIS,root_container,ROOT_CONTAINER,1,0,0,1778724023477
1,POLARIS,root,PRINCIPAL,1,0,6344441740302737540,1778724023496
2,POLARIS,spark-service,PRINCIPAL,1,0,5248648849929431605,1778731831759
3,POLARIS,service_admin,PRINCIPAL_ROLE,1,0,2492644736706797330,1778724023588
4,POLARIS,data-engineer,PRINCIPAL_ROLE,1,0,8351745351947001431,1778725724061
5,POLARIS,test-catalog,CATALOG,1,0,8266428657270988246,1778733122854
6,POLARIS,catalog_admin,NAMESPACE,1,8266428657270988246,4937705904896732289,1778733122870
7,POLARIS,data-engineer-role,NAMESPACE,1,8266428657270988246,3268010189368423518,1778733214005
8,POLARIS,test_namespace,TABLE_LIKE,1,8266428657270988246,7566770789005112158,1778733256225
9,POLARIS,test_table,CATALOG_ROLE,2,7566770789005112158,7075209999268783556,1778741414595


In [4]:
# Cell 4 — Inspect specific entity properties
print('=' * 60)
print('ENTITY PROPERTIES INSPECTION')
print('=' * 60)

# Catalog entity - shows metadata_location
print('\n📚 Catalog entities (properties + internal_properties):')
df_catalog = pg_query("""
    SELECT name, entity_version, properties, internal_properties
    FROM polaris_schema.entities
    WHERE type_code = 4
""")
for _, row in df_catalog.iterrows():
    print(f'\n  Catalog: {row["name"]} (v{row["entity_version"]})')
    print(f'  properties:          {json.dumps(row["properties"], indent=4)}')
    print(f'  internal_properties: {json.dumps(row["internal_properties"], indent=4)}')

# Principal entity - shows client_id
print('\n👤 Principal entities:')
df_principal = pg_query("""
    SELECT name, entity_version, properties, internal_properties
    FROM polaris_schema.entities
    WHERE type_code = 2
""")
for _, row in df_principal.iterrows():
    print(f'\n  Principal: {row["name"]} (v{row["entity_version"]})')
    print(f'  internal_properties: {json.dumps(row["internal_properties"], indent=4)}')

# Table entity - shows metadata_location
print('\n📊 Table entities:')
df_table = pg_query("""
    SELECT name, entity_version, properties, internal_properties
    FROM polaris_schema.entities
    WHERE type_code = 6
""")
for _, row in df_table.iterrows():
    print(f'\n  Table: {row["name"]} (v{row["entity_version"]})')
    print(f'  properties:          {json.dumps(row["properties"], indent=4)}')
    print(f'  internal_properties: {json.dumps(row["internal_properties"], indent=4)}')

ENTITY PROPERTIES INSPECTION

📚 Catalog entities (properties + internal_properties):

  Catalog: test-catalog (v1)
  properties:          {
    "s3.endpoint": "http://benchmarks-minio.datahub-hynix.svc.cluster.local:9000",
    "s3.path-style-access": "true",
    "default-base-location": "s3://data-catalog-bucket/test-catalog"
}
  internal_properties: {
    "catalogType": "INTERNAL",
    "storage_configuration_info": "{\"@type\":\"AwsStorageConfigurationInfo\",\"allowedLocations\":[\"s3://data-catalog-bucket/test-catalog\",\"s3://data-catalog-bucket/\"],\"roleARN\":\"arn:aws:iam::123456789012:role/dummy\",\"pathStyleAccess\":true,\"fileIoImplClassName\":\"org.apache.iceberg.aws.s3.S3FileIO\",\"storageType\":\"S3\"}"
}

👤 Principal entities:

  Principal: root (v1)
  internal_properties: {
    "client_id": "root"
}

  Principal: spark-service (v1)
  internal_properties: {
    "client_id": "61f86631c73fcae9"
}

📊 Table entities:

  Table: test_namespace (v1)
  properties:          {
    "

## grant_records table
RBAC 권한 부여 정보 — 누가 어떤 엔티티에 어떤 권한을 가지는지

In [5]:
# Cell 5 — grant_records table
print('=' * 60)
print('GRANT_RECORDS TABLE')
print('=' * 60)

# Privilege code mapping
PRIVILEGE_CODES = {
    1:  'CATALOG_MANAGE_ACCESS',
    2:  'CATALOG_MANAGE_CONTENT',
    3:  'CATALOG_MANAGE_METADATA',
    4:  'CATALOG_READ_PROPERTIES',
    5:  'CATALOG_WRITE_PROPERTIES',
    6:  'NAMESPACE_CREATE',
    7:  'NAMESPACE_LIST',
    8:  'NAMESPACE_READ_PROPERTIES',
    9:  'NAMESPACE_WRITE_PROPERTIES',
    10: 'NAMESPACE_FULL_METADATA',
    11: 'SERVICE_MANAGE_ACCESS',
    12: 'PRINCIPAL_ROLE_USAGE',
    13: 'CATALOG_ROLE_USAGE',
    14: 'TABLE_READ_DATA',
    15: 'TABLE_WRITE_DATA',
    16: 'TABLE_READ_PROPERTIES',
    17: 'TABLE_WRITE_PROPERTIES',
    18: 'TABLE_FULL_METADATA',
    19: 'VIEW_READ_PROPERTIES',
    20: 'VIEW_WRITE_PROPERTIES',
    21: 'VIEW_FULL_METADATA',
    22: 'VIEW_CREATE',
    23: 'TABLE_CREATE',
}

df_grants = pg_query("""
    SELECT 
        g.realm_id,
        g.securable_id,
        g.grantee_id,
        g.privilege_code,
        s.name AS securable_name,
        s.type_code AS securable_type,
        e.name AS grantee_name,
        e.type_code AS grantee_type
    FROM polaris_schema.grant_records g
    LEFT JOIN polaris_schema.entities s 
        ON g.securable_id = s.id AND g.realm_id = s.realm_id
    LEFT JOIN polaris_schema.entities e 
        ON g.grantee_id = e.id AND g.realm_id = e.realm_id
    ORDER BY g.privilege_code
""")

TYPE_CODES = {
    1: 'ROOT_CONTAINER', 2: 'PRINCIPAL', 3: 'PRINCIPAL_ROLE',
    4: 'CATALOG', 5: 'NAMESPACE', 6: 'TABLE', 7: 'CATALOG_ROLE'
}

df_grants['privilege']      = df_grants['privilege_code'].map(PRIVILEGE_CODES)
df_grants['securable_type'] = df_grants['securable_type'].map(TYPE_CODES)
df_grants['grantee_type']   = df_grants['grantee_type'].map(TYPE_CODES)

print(f'\n📊 Total grants: {len(df_grants)}')
print('\n📋 Grant records (human readable):')
display(df_grants[['grantee_name', 'grantee_type', 'privilege', 'securable_name', 'securable_type']])

print('\n💡 Reading: grantee has privilege on securable')
print('   Example: data-engineer-role has CATALOG_MANAGE_CONTENT on test-catalog')

GRANT_RECORDS TABLE

📊 Total grants: 8

📋 Grant records (human readable):


,grantee_name,grantee_type,privilege,securable_name,securable_type
0,service_admin,PRINCIPAL_ROLE,CATALOG_MANAGE_ACCESS,root_container,ROOT_CONTAINER
1,catalog_admin,NAMESPACE,CATALOG_MANAGE_CONTENT,test-catalog,CATALOG
2,data-engineer,PRINCIPAL_ROLE,CATALOG_MANAGE_METADATA,data-engineer-role,NAMESPACE
3,service_admin,PRINCIPAL_ROLE,CATALOG_MANAGE_METADATA,catalog_admin,NAMESPACE
4,root,PRINCIPAL,CATALOG_READ_PROPERTIES,service_admin,PRINCIPAL_ROLE
5,spark-service,PRINCIPAL,CATALOG_READ_PROPERTIES,data-engineer,PRINCIPAL_ROLE
6,catalog_admin,NAMESPACE,NaN,test-catalog,CATALOG
7,data-engineer-role,NAMESPACE,NaN,test-catalog,CATALOG



💡 Reading: grantee has privilege on securable
   Example: data-engineer-role has CATALOG_MANAGE_CONTENT on test-catalog


## version table
메타스토어 스키마 버전 관리

In [6]:
# Cell 6 — version table
print('=' * 60)
print('VERSION TABLE')
print('=' * 60)

df_version = pg_query('SELECT * FROM polaris_schema.version')
display(df_version)
print('\n💡 This tracks the metastore schema version')
print('   Used for schema migrations when Polaris upgrades')

VERSION TABLE


,version_key,version_value
0,version,3



💡 This tracks the metastore schema version
   Used for schema migrations when Polaris upgrades


In [7]:
# Cell 7 — Entity parent-child hierarchy
print('=' * 60)
print('ENTITY HIERARCHY')
print('=' * 60)

TYPE_CODES = {
    1: 'ROOT_CONTAINER', 2: 'PRINCIPAL', 3: 'PRINCIPAL_ROLE',
    4: 'CATALOG', 5: 'NAMESPACE', 6: 'TABLE', 7: 'CATALOG_ROLE'
}

df_all = pg_query("""
    SELECT id, parent_id, name, type_code
    FROM polaris_schema.entities
    ORDER BY type_code, id
""")

# Build hierarchy map
id_to_name = dict(zip(df_all['id'], df_all['name']))

print('\n🌳 Entity parent-child relationships:')
for _, row in df_all.iterrows():
    parent_name = id_to_name.get(row['parent_id'], 'ROOT')
    entity_type = TYPE_CODES.get(row['type_code'], '?')
    if row['id'] == row['parent_id']:
        print(f'  [{entity_type}] {row["name"]} (root)')
    else:
        print(f'  [{entity_type}] {row["name"]} → parent: {parent_name}')

ENTITY HIERARCHY

🌳 Entity parent-child relationships:
  [ROOT_CONTAINER] root_container (root)
  [PRINCIPAL] spark-service → parent: root_container
  [PRINCIPAL] root → parent: root_container
  [PRINCIPAL_ROLE] service_admin → parent: root_container
  [PRINCIPAL_ROLE] data-engineer → parent: root_container
  [CATALOG] test-catalog → parent: root_container
  [NAMESPACE] data-engineer-role → parent: test-catalog
  [NAMESPACE] catalog_admin → parent: test-catalog
  [TABLE] test_namespace → parent: test-catalog
  [CATALOG_ROLE] test_table → parent: test_namespace
  [?] entityCleanup_1532346270757427849 → parent: root_container


In [8]:
# Cell 8 — Compare API vs PostgreSQL
print('=' * 60)
print('API vs PostgreSQL COMPARISON')
print('=' * 60)

# Get principal from API
api_principals = requests.get(
    f'{POLARIS_URL}/api/management/v1/principals',
    headers={'Authorization': f'Bearer {TOKEN}', 'Polaris-Realm': REALM}
).json().get('principals', [])

# Get principal from PostgreSQL
pg_principals = pg_query("""
    SELECT name, entity_version, create_timestamp, last_update_timestamp
    FROM polaris_schema.entities
    WHERE type_code = 2
""")

print('\n📡 From Polaris API:')
for p in api_principals:
    print(f'  name={p["name"]} entityVersion={p["entityVersion"]} createTimestamp={p["createTimestamp"]}')

print('\n🗄️  From PostgreSQL directly:')
display(pg_principals)

print('\n✅ API and PostgreSQL data match — Polaris reads directly from PostgreSQL')

API vs PostgreSQL COMPARISON

📡 From Polaris API:
  name=root entityVersion=1 createTimestamp=1778724023496
  name=spark-service entityVersion=1 createTimestamp=1778731831759

🗄️  From PostgreSQL directly:


,name,entity_version,create_timestamp,last_update_timestamp
0,root,1,1778724023496,1778724023496
1,spark-service,1,1778731831759,1778731831759



✅ API and PostgreSQL data match — Polaris reads directly from PostgreSQL


## 관찰 결과 (Observations)

- `entities` 테이블은 모든 엔티티 타입을 type_code로 구분하여 단일 테이블에 저장
- `grant_records` 테이블은 RBAC 권한을 privilege_code로 표현
- `version` 테이블은 스키마 마이그레이션 관리
- Polaris API 응답값과 PostgreSQL 직접 조회 결과가 동일함
- `internal_properties` JSONB에 metadata_location 등 내부 정보 저장

## 핵심 학습 포인트 (Key Takeaways)

1. Metastore = Polaris의 모든 엔티티를 저장하는 PostgreSQL 스키마
2. In-memory는 개발용, Relational JDBC(PostgreSQL)는 운영용
3. Bootstrap이 스키마 자동 생성 + 초기 realm/principal 삽입
4. entities 테이블 하나가 모든 타입을 type_code로 구분
5. grant_records가 RBAC의 실제 저장소
